## Análise de Documentos PDF: Explorando Características e Relações

#### Participantes

- Anne Fernandes da Costa Oliveira
- Matheus da Silva Nunes
- Thiago Henrique Menêses Bezerra


In [1]:
# %%
# Imports e configurações gerais
import random
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import fitz  # PyMuPDF
import nltk
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display
from nltk.corpus import stopwords
from plotly.subplots import make_subplots
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, silhouette_score
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

# Configuração de template e paleta de cores aprimorada
pio.templates.default = "plotly_white"

# Nova paleta de cores mais contrastante e acessível
MAIN_COLOR = "#2E86C1"  # Azul principal
SEC_COLOR = "#E74C3C"  # Vermelho secundário
HIGHLIGHT_COLOR = "#27AE60"  # Verde destaque
CLUSTER_PALETTE = px.colors.qualitative.Bold  # Paleta qualitativa para clusters

# Configuração padrão para gráficos
FIGURE_CONFIG = {
    "width": 900,
    "height": 600,
    "font": {"family": "Arial", "size": 12},
    "margin": {"l": 80, "r": 80, "t": 100, "b": 80},
}

# Paths e inicializações
DATASET_PATH = Path("data/processed/dataset.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Configuração de stopwords em Português
nltk.download("stopwords", quiet=True)
PT_STOPWORDS: List[str] = stopwords.words("portuguese")

# Fixar seed para reprodutibilidade
random.seed(42)
np.random.seed(42)


### Motivação

Os documentos digitais, especialmente PDFs, são um formato central na disseminação de informações em ambientes corporativos,
governamentais e acadêmicos. Compreender as características desses documentos e as relações entre seus atributos físicos
(como tamanho e número de páginas) e seu conteúdo textual pode fornecer insights valiosos para:

- Otimização de sistemas de armazenamento
- Melhoria de algoritmos de busca e recuperação
- Identificação de padrões em grandes coleções documentais
- Detecção de anomalias ou documentos atípicos

### Objetivos

Esta análise tem como objetivos principais:

1. Caracterizar a distribuição do número de páginas em documentos PDF
2. Identificar a relação entre número de páginas e tamanho do arquivo
3. Descobrir agrupamentos naturais (clusters) baseados em características físicas
4. Identificar padrões temáticos nos conteúdos textuais
5. Explorar possíveis relações entre as características físicas e o conteúdo dos documentos

O conhecimento gerado pode ser utilizado para melhorar sistemas de gestão documental,
otimizar processos de indexação e promover um entendimento mais profundo da estrutura
informacional presente em coleções de documentos digitais.


## Descrição do Dataset

O dataset utilizado nesta análise contém informações sobre documentos PDF, com as seguintes características:

- **URL**: Caminho/localização do arquivo PDF
- **nato-digital**: Flag indicando se o documento é nativo digital (não escaneado)
- **numero-de-paginas**: Quantidade de páginas do documento
- **megabytes**: Tamanho do arquivo em MB

Para esta análise, focamos apenas nos documentos nativos digitais (nato-digital=True) e
que estão disponíveis no sistema de arquivos local, garantindo a possibilidade de extrair
e analisar seu conteúdo textual.


In [2]:
# %%
# =============================================================================
# 2. DADOS USADOS: DESCRIÇÃO DO DATASET E EXPLORAÇÃO INICIAL
# =============================================================================


def load_and_preprocess(path: Path) -> pd.DataFrame:
    """
    Carrega o CSV e retorna DataFrame com PDFs nativos disponíveis em disco.

    Args:
        path: Caminho para o arquivo CSV de dados.

    Returns:
        pd.DataFrame: Dados com coluna 'path' para cada arquivo PDF.
    """
    df = pd.read_csv(
        path,
        usecols=["URL", "nato-digital", "numero-de-paginas", "megabytes"],
        dtype={
            "URL": str,
            "nato-digital": bool,
            "numero-de-paginas": int,
            "megabytes": float,
        },
    )
    df = df[df["nato-digital"]].reset_index(drop=True)
    df["path"] = df["URL"].str.replace("\\", "/", regex=False).map(Path)
    df = df[df["path"].map(lambda p: p.is_file())].reset_index(drop=True)
    return df


# Carrega dados e exibe quantidade de documentos digitais válidos
df = load_and_preprocess(DATASET_PATH)

# Criando um resumo das características básicas
basic_stats = {
    "Total de documentos": len(df),
    "Tamanho total (MB)": df["megabytes"].sum(),
    "Total de páginas": df["numero-de-paginas"].sum(),
    "Tamanho médio (MB)": df["megabytes"].mean(),
    "Páginas médias por documento": df["numero-de-paginas"].mean(),
}

# Exibindo as estatísticas básicas em uma tabela formatada
stats_md = "| Característica | Valor |\n|---------------|-------|\n"
for key, value in basic_stats.items():
    if isinstance(value, int):
        formatted_value = f"{value:,}"
    else:
        formatted_value = f"{value:.2f}"
    stats_md += f"| {key} | {formatted_value} |\n"

display(Markdown(stats_md))

# Criando um gráfico scatter básico para visualizar a coleção
fig = px.scatter(
    df,
    x="numero-de-paginas",
    y="megabytes",
    title="<b>Visão Geral da Coleção: Relação entre Número de Páginas e Tamanho</b>",
    labels={"numero-de-paginas": "Número de Páginas", "megabytes": "Tamanho (MB)"},
    color_discrete_sequence=[MAIN_COLOR],
    opacity=0.7,
)

fig.update_layout(
    title_x=0.5,
    **FIGURE_CONFIG,
    xaxis_title="<b>Número de Páginas</b>",
    yaxis_title="<b>Tamanho (MB)</b>",
)

fig.show()

| Característica | Valor |
|---------------|-------|
| Total de documentos | 4,134 |
| Tamanho total (MB) | 3814.90 |
| Total de páginas | 121485.00 |
| Tamanho médio (MB) | 0.92 |
| Páginas médias por documento | 29.39 |


**Conclusões da Exploração Inicial:**

Esta coleção consiste em documentos PDF nativos digitais. Observa-se uma variação considerável no tamanho e número de páginas entre os documentos, sugerindo uma coleção diversificada. O gráfico de dispersão inicial indica uma possível relação linear entre o número de páginas e o tamanho do arquivo, que será explorada em detalhes na análise de regressão.


## Etapas de Pré-processamento

O pré-processamento dos dados envolve as seguintes etapas:

1. **Filtragem inicial**: Selecionar apenas documentos nativos digitais (já realizado durante o carregamento)
2. **Verificação de disponibilidade**: Confirmar que os arquivos estão no sistema (já realizado durante o carregamento)
3. **Extração de texto**: Obter o conteúdo textual dos PDFs para análise temática
4. **Normalização de variáveis**: Escalonar variáveis numéricas para análise de clusters
5. **Processamento de texto**: Aplicar técnicas de NLP como remoção de stopwords e vetorização TF-IDF

A seguir, implementamos as funções necessárias para estas etapas.


In [3]:
# %%
# =============================================================================
# 3. PRÉ-PROCESSAMENTO: LIMPEZA E TRANSFORMAÇÃO DOS DADOS
# =============================================================================


def extract_texts(df: pd.DataFrame, max_pages: Optional[int] = None) -> List[str]:
    """
    Extrai texto de PDFs, limitado a `max_pages` por documento.

    Args:
        df: DataFrame com coluna 'path' apontando para PDFs.
        max_pages: Máximo de páginas a extrair (todas se None).

    Returns:
        List[str]: Lista de conteúdo textual de cada PDF.
    """
    texts: List[str] = []
    for pdf_path in tqdm(df["path"], desc="Extraindo textos"):
        with fitz.open(pdf_path) as doc:
            total = doc.page_count
            pages = list(range(total))
            if max_pages and max_pages > 0:
                pages = random.sample(pages, min(max_pages, total))
            text = "\n".join(doc.load_page(i).get_text() for i in pages)
        texts.append(text)
    return texts


### Decisões de Pré-processamento

1. **Limitação de páginas**: Para documentos muito extensos, optamos por extrair um subconjunto aleatório de páginas (até 3 por documento) para a análise de conteúdo. Isso equilibra a representatividade do conteúdo com a eficiência computacional.

2. **Normalização para clustering**: Para a análise de clusters, normalizamos as variáveis numéricas (número de páginas e tamanho) usando StandardScaler, garantindo que ambas as características contribuam igualmente para a formação dos clusters.

3. **Processamento de texto**: Para análise de conteúdo, aplicamos as seguintes transformações:
   - Remoção de stopwords em português
   - Filtragem de termos muito raros (presentes em menos de 20 documentos) ou muito comuns (presentes em mais de 80% dos documentos)
   - Vetorização TF-IDF para capturar a importância relativa dos termos

Estas decisões de pré-processamento visam garantir uma análise robusta e significativa, considerando tanto as características físicas quanto o conteúdo dos documentos.


## Pergunta 1: Qual é a distribuição do número de páginas nos documentos?


In [4]:
# %%
# Pergunta 1: Qual é a distribuição do número de páginas nos documentos?
# =============================================================================


def analyze_distribution(df: pd.DataFrame) -> Dict[str, Any]:
    """
    Exibe estatísticas e gráfico interativo da distribuição de número de páginas.

    Args:
        df: DataFrame contendo coluna 'numero-de-paginas'.

    Returns:
        Dict[str, Any]: Dicionário com estatísticas-chave para uso em outras funções
    """
    pages = df["numero-de-paginas"].dropna()
    n = len(pages)
    mean = pages.mean()
    median = pages.median()
    std = pages.std(ddof=1)
    q1, q3 = pages.quantile([0.25, 0.75])
    iqr = q3 - q1
    ci_low, ci_high = stats.t.interval(0.95, df=n - 1, loc=mean, scale=stats.sem(pages))
    skewness = stats.skew(pages)
    kurtosis = stats.kurtosis(pages)

    # Classificação da distribuição
    distribution_type = "aproximadamente normal"
    if skewness > 1:
        distribution_type = "assimétrica à direita (cauda longa positiva)"
    elif skewness < -1:
        distribution_type = "assimétrica à esquerda (cauda longa negativa)"

    # Detecção de outliers usando IQR
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = pages[(pages < lower_bound) | (pages > upper_bound)]

    # Definição de estatísticas de referência e suas cores
    references = [
        {"value": mean, "name": "Média", "color": MAIN_COLOR},
        {"value": median, "name": "Mediana", "color": SEC_COLOR},
        {"value": q1, "name": "Q1", "color": HIGHLIGHT_COLOR},
        {"value": q3, "name": "Q3", "color": HIGHLIGHT_COLOR},
    ]

    # Criação do histograma aprimorado
    fig = px.histogram(
        pages,
        nbins=30,
        marginal="box",
        title="<b>Distribuição do Número de Páginas nos Documentos</b>",
        labels={"value": "Número de Páginas", "count": "Frequência"},
        color_discrete_sequence=[MAIN_COLOR],
        opacity=0.8,
    )

    # Adição de linhas de referência com cores distintas
    for ref in references:
        fig.add_vline(
            x=ref["value"],
            line_dash="dash",
            line_color=ref["color"],
            line_width=2,
            annotation_text=f"{ref['name']}: {ref['value']:.1f}",
            annotation_position="top right",
            annotation_font_color=ref["color"],
            annotation_font_size=12,
        )

    fig.update_xaxes(title_text="<b>Número de Páginas</b>", title_font=dict(size=14))
    fig.update_yaxes(title_text="<b>Frequência</b>", title_font=dict(size=14))

    fig.update_layout(
        showlegend=False,
        title_x=0.5,
        title_font=dict(size=16),
        **FIGURE_CONFIG,
        hovermode="x unified",
    )

    # Adição de anotações para destacar outliers e padrões
    max_pages = pages.max()
    fig.add_annotation(
        x=max_pages,
        y=0,
        text=f"Máximo: {max_pages}",
        showarrow=True,
        arrowhead=2,
        arrowcolor=SEC_COLOR,
        ax=-40,
        ay=-40,
    )

    fig.show()

    # Exibição do resumo estatístico
    resumo = (
        f"**Resumo da Distribuição**\n\n"
        f"- **N** = {n:,}\n"
        f"- **Média** = {mean:.2f} (IC95%: {ci_low:.2f}–{ci_high:.2f})\n"
        f"- **Mediana** = {median:.2f}\n"
        f"- **IQR** = {iqr:.2f}\n"
        f"- **Desvio Padrão (σ)** = {std:.2f}\n"
        f"- **Mínimo** = {pages.min()}\n"
        f"- **Máximo** = {pages.max()}\n"
        f"- **Assimetria** = {skewness:.2f}\n"
        f"- **Curtose** = {kurtosis:.2f}\n"
        f"- **Outliers detectados** = {len(outliers)} documentos\n"
    )
    display(Markdown(resumo))

    # Interpretação dos resultados
    interpretation = (
        f"**Interpretação da Distribuição:**\n\n"
        f"A distribuição do número de páginas é {distribution_type}. "
        f"A diferença entre média ({mean:.2f}) e mediana ({median:.2f}) "
        f"{'indica presença de valores extremos que puxam a média para cima' if mean > median else 'é pequena, sugerindo poucos valores extremos'}"
        f". Os outliers representam {len(outliers) / n * 100:.1f}% da coleção."
    )
    display(Markdown(interpretation))

    # Retornar estatísticas para uso em outras funções
    return {
        "mean": mean,
        "median": median,
        "std": std,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "skewness": skewness,
        "outliers": outliers,
        "distribution_type": distribution_type,
    }


distribution_stats = analyze_distribution(df)

**Resumo da Distribuição**

- **N** = 4,134
- **Média** = 29.39 (IC95%: 26.02–32.75)
- **Mediana** = 3.00
- **IQR** = 13.00
- **Desvio Padrão (σ)** = 110.33
- **Mínimo** = 1
- **Máximo** = 2816
- **Assimetria** = 9.41
- **Curtose** = 140.53
- **Outliers detectados** = 553 documentos


**Interpretação da Distribuição:**

A distribuição do número de páginas é assimétrica à direita (cauda longa positiva). A diferença entre média (29.39) e mediana (3.00) indica presença de valores extremos que puxam a média para cima. Os outliers representam 13.4% da coleção.

## Pergunta 2: Existe uma relação linear entre o número de páginas e o tamanho do arquivo?


In [5]:
# %%
# Pergunta 2: Existe uma relação linear entre o número de páginas e o tamanho do arquivo?
# =============================================================================


def analyze_regression(df: pd.DataFrame) -> Dict[str, Any]:
    """
    Ajusta e plota regressão linear entre número de páginas e tamanho em MB.

    Args:
        df: DataFrame contendo 'numero-de-paginas' e 'megabytes'.

    Returns:
        Dict[str, Any]: Dicionário com estatísticas-chave da regressão
    """
    X = df[["numero-de-paginas"]].values
    y = df["megabytes"].values
    model = LinearRegression().fit(X, y)
    slope = model.coef_[0]
    intercept = model.intercept_
    r2 = r2_score(y, model.predict(X))

    # Cálculo de métricas adicionais
    y_pred = model.predict(X)
    residuals = y - y_pred
    mse = np.mean(residuals**2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(residuals))

    # Normalização do RMSE para comparabilidade
    normalized_rmse = rmse / (y.max() - y.min()) * 100

    # Ordenação para traçar a linha de regressão
    idx = np.argsort(X.flatten())
    X_sorted = X[idx].flatten()
    y_sorted = y_pred[idx]

    # Gráfico aprimorado com plotly
    fig = go.Figure()

    # Pontos de dados
    fig.add_trace(
        go.Scatter(
            x=df["numero-de-paginas"],
            y=df["megabytes"],
            mode="markers",
            name="Documentos",
            marker=dict(
                color=MAIN_COLOR,
                size=8,
                opacity=0.7,
                line=dict(width=1, color="white"),
            ),
            hovertemplate="<b>Páginas:</b> %{x}<br><b>Tamanho:</b> %{y:.2f} MB<extra></extra>",
        )
    )

    # Linha de regressão
    fig.add_trace(
        go.Scatter(
            x=X_sorted,
            y=y_sorted,
            mode="lines",
            name=f"Regressão Linear (R² = {r2:.3f})",
            line=dict(color=SEC_COLOR, width=3),
            hovertemplate="<b>Páginas:</b> %{x}<br><b>Tamanho estimado:</b> %{y:.2f} MB<extra></extra>",
        )
    )

    # Adicionar intervalo de confiança (opcional)
    n = len(X)
    t_critical = stats.t.ppf(0.975, df=n - 2)  # Para 95% de confiança
    x_mean = X.mean()
    s_xx = ((X - x_mean) ** 2).sum()

    # Limites do intervalo de confiança
    conf_interval = []
    for xi in X_sorted:
        se = np.sqrt(mse * (1 / n + (xi - x_mean) ** 2 / s_xx))
        conf_lower = y_sorted[np.where(X_sorted == xi)[0][0]] - t_critical * se
        conf_upper = y_sorted[np.where(X_sorted == xi)[0][0]] + t_critical * se
        conf_interval.append((conf_lower, conf_upper))

    conf_interval = np.array(conf_interval)

    # Traçar intervalo de confiança
    fig.add_trace(
        go.Scatter(
            x=np.concatenate([X_sorted, X_sorted[::-1]]),
            y=np.concatenate([conf_interval[:, 0], conf_interval[:, 1][::-1]]),
            fill="toself",
            fillcolor="rgba(231, 76, 60, 0.2)",
            line=dict(color="rgba(255, 255, 255, 0)"),
            name="Intervalo de Confiança (95%)",
            showlegend=True,
        )
    )

    # Configuração do layout
    fig.update_layout(
        title="<b>Regressão Linear: Relação entre Número de Páginas e Tamanho do Arquivo</b>",
        xaxis_title="<b>Número de Páginas</b>",
        yaxis_title="<b>Tamanho do Arquivo (MB)</b>",
        hovermode="closest",
        legend=dict(
            x=0.02,
            y=0.98,
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor="lightgray",
            borderwidth=1,
        ),
        title_x=0.5,
        title_font=dict(size=16),
        **FIGURE_CONFIG,
    )

    # Anotação da equação
    equation = f"y = {slope:.4f}x + {intercept:.4f}"
    fig.add_annotation(
        x=0.02,
        y=0.85,
        xref="paper",
        yref="paper",
        text=f"<b>Equação:</b> {equation}<br><b>R²:</b> {r2:.3f} ({r2 * 100:.1f}% da variância explicada)",
        showarrow=False,
        font=dict(size=14),
        align="left",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="lightgray",
        borderwidth=1,
        borderpad=4,
    )

    fig.show()

    # Exibição do resumo da regressão
    reg_text = (
        f"**Resumo do Modelo de Regressão Linear**\n\n"
        f"- **Equação da regressão:** y = {slope:.4f}x + {intercept:.4f}  \n"
        f"- **R²:** {r2:.3f} ({r2 * 100:.1f}% da variância explicada)\n"
        f"- **Erro Quadrático Médio (MSE):** {mse:.4f}\n"
        f"- **Raiz do Erro Quadrático Médio (RMSE):** {rmse:.4f} MB\n"
        f"- **RMSE Normalizado:** {normalized_rmse:.2f}%\n"
        f"- **Erro Absoluto Médio (MAE):** {mae:.4f} MB\n"
    )
    display(Markdown(reg_text))

    # Interpretação dos resultados
    strength = "forte" if r2 > 0.7 else "moderada" if r2 > 0.4 else "fraca"
    fit_quality = (
        "excelente"
        if normalized_rmse < 10
        else "bom"
        if normalized_rmse < 20
        else "moderado"
        if normalized_rmse < 30
        else "fraco"
    )

    interpretation = (
        f"**Interpretação da Regressão:**\n\n"
        f"Existe uma relação linear {strength} entre o número de páginas e o tamanho do arquivo. "
        f"Cada página adicional contribui, em média, com {slope:.4f} MB para o tamanho total do arquivo. "
        f"O modelo tem um ajuste {fit_quality} aos dados (RMSE normalizado: {normalized_rmse:.2f}%). "
        f"O número de páginas explica {r2 * 100:.1f}% da variabilidade no tamanho dos arquivos, "
        f"sugerindo que {'outros fatores como complexidade do conteúdo, presença de imagens ou formatação também influenciam significativamente o tamanho' if r2 < 0.7 else 'o número de páginas é o fator dominante que determina o tamanho do arquivo'}."
    )
    display(Markdown(interpretation))

    # Retornar estatísticas para uso em outras funções
    return {
        "slope": slope,
        "intercept": intercept,
        "r2": r2,
        "rmse": rmse,
        "normalized_rmse": normalized_rmse,
        "model": model,
    }


regression_stats = analyze_regression(df)

**Resumo do Modelo de Regressão Linear**

- **Equação da regressão:** y = 0.0092x + 0.6520  
- **R²:** 0.110 (11.0% da variância explicada)
- **Erro Quadrático Médio (MSE):** 8.3575
- **Raiz do Erro Quadrático Médio (RMSE):** 2.8909 MB
- **RMSE Normalizado:** 4.27%
- **Erro Absoluto Médio (MAE):** 0.9057 MB


**Interpretação da Regressão:**

Existe uma relação linear fraca entre o número de páginas e o tamanho do arquivo. Cada página adicional contribui, em média, com 0.0092 MB para o tamanho total do arquivo. O modelo tem um ajuste excelente aos dados (RMSE normalizado: 4.27%). O número de páginas explica 11.0% da variabilidade no tamanho dos arquivos, sugerindo que outros fatores como complexidade do conteúdo, presença de imagens ou formatação também influenciam significativamente o tamanho.

## Pergunta 3: Quantos grupos naturais existem entre os documentos com base em suas características físicas?


In [6]:
# %%
# Pergunta 3: Quantos grupos naturais existem entre os documentos com base em suas características físicas?
# =============================================================================


def compute_optimal_k(
    X: np.ndarray,
    max_k: int,
    title: str = "Definição do Número Ideal de Clusters",
) -> int:
    """
    Calcula k ótimo combinando inércia e coeficiente de silhueta.

    Args:
        X: Dados escalados.
        max_k: Valor máximo de k a testar.
        title: Título do gráfico.

    Returns:
        int: Valor de k ótimo.
    """
    ks = list(range(2, max_k + 1))
    inertias: List[float] = []
    silhouettes: List[float] = []

    # Cálculo das métricas para diferentes valores de k
    for k in tqdm(ks, desc="Calculando métricas para valores de k"):
        km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
        inertias.append(km.inertia_)
        silhouettes.append(silhouette_score(X, km.labels_))

    # Normalização das métricas para comparação
    inv_inertia = 1 - (np.array(inertias) - min(inertias)) / (
        max(inertias) - min(inertias)
    )
    sil_norm = (np.array(silhouettes) - min(silhouettes)) / (
        max(silhouettes) - min(silhouettes)
    )

    # Cálculo da pontuação combinada
    scores = (inv_inertia + sil_norm) / 2
    k_opt = ks[int(np.argmax(scores))]

    # Criação do gráfico aprimorado
    fig = go.Figure()

    # Adição das métricas com eixos distintos
    fig.add_trace(
        go.Scatter(
            x=ks,
            y=inertias,
            mode="lines+markers",
            name="Inércia",
            marker=dict(size=8, color=MAIN_COLOR),
            line=dict(width=2, color=MAIN_COLOR),
            hovertemplate="<b>k:</b> %{x}<br><b>Inércia:</b> %{y:.2f}<extra></extra>",
        )
    )

    fig.add_trace(
        go.Scatter(
            x=ks,
            y=silhouettes,
            mode="lines+markers",
            name="Coeficiente de Silhueta",
            marker=dict(size=8, color=SEC_COLOR),
            line=dict(width=2, color=SEC_COLOR),
            yaxis="y2",
            hovertemplate="<b>k:</b> %{x}<br><b>Silhueta:</b> %{y:.3f}<extra></extra>",
        )
    )

    # Adicionando a pontuação combinada
    fig.add_trace(
        go.Scatter(
            x=ks,
            y=scores,
            mode="lines+markers",
            name="Pontuação Combinada",
            marker=dict(size=8, color=HIGHLIGHT_COLOR),
            line=dict(width=2, color=HIGHLIGHT_COLOR, dash="dot"),
            yaxis="y3",
            hovertemplate="<b>k:</b> %{x}<br><b>Pontuação:</b> %{y:.3f}<extra></extra>",
        )
    )

    # Adição da linha indicativa do k ótimo
    fig.add_vline(
        x=k_opt,
        line=dict(dash="dot", width=2, color=HIGHLIGHT_COLOR),
        annotation=dict(
            text=f"<b>k ótimo = {k_opt}</b>",
            font=dict(size=14, color=HIGHLIGHT_COLOR),
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor=HIGHLIGHT_COLOR,
            borderwidth=1,
            borderpad=4,
            x=k_opt + 0.5,
            xanchor="left",
            y=0.95,
        ),
    )

    # Configuração do layout com eixos duplos
    fig.update_layout(
        title=f"<b>{title}</b>",
        xaxis=dict(
            title=dict(
                text="<b>Número de Clusters (k)</b>",
            ),
            tickmode="linear",
            dtick=1,
        ),
        yaxis=dict(
            title=dict(
                text="<b>Inércia</b> (quanto menor, melhor)",
                font=dict(color=MAIN_COLOR),
            ),
            tickfont=dict(color=MAIN_COLOR),
        ),
        yaxis2=dict(
            title=dict(
                text="<b>Coeficiente de Silhueta</b> (quanto maior, melhor)",
                font=dict(color=SEC_COLOR),
            ),
            tickfont=dict(color=SEC_COLOR),
            overlaying="y",
            side="right",
        ),
        yaxis3=dict(
            title=dict(
                text="<b>Pontuação Combinada</b>", font=dict(color=HIGHLIGHT_COLOR)
            ),
            tickfont=dict(color=HIGHLIGHT_COLOR),
            overlaying="y",
            side="right",
            showgrid=False,
            zeroline=False,
            showticklabels=False,
        ),
        legend=dict(
            x=0.01,
            y=0.99,
            bgcolor="rgba(255,255,255,0.8)",
            bordercolor="lightgray",
            borderwidth=1,
        ),
        title_x=0.5,
        title_font=dict(size=16),
        **FIGURE_CONFIG,
        hovermode="x unified",
    )

    # Adição de anotação explicativa
    fig.add_annotation(
        x=0.5,
        y=-0.15,
        xref="paper",
        yref="paper",
        text="<b>Nota:</b> O k ótimo representa o equilíbrio entre a minimização da inércia (variação intra-cluster) e a maximização do coeficiente de silhueta (separação entre clusters).",
        showarrow=False,
        font=dict(size=12),
        align="center",
    )

    fig.show()

    # Tabela de métricas para cada k
    metric_df = pd.DataFrame(
        {
            "k": ks,
            "Inércia": inertias,
            "Silhueta": silhouettes,
            "Pontuação Combinada": scores,
        }
    )

    display(Markdown("**Métricas por Número de Clusters**"))
    display(
        metric_df.style.highlight_max(
            subset=["Silhueta", "Pontuação Combinada"], color="#D5F5E3"
        )
        .highlight_min(subset=["Inércia"], color="#D5F5E3")
        .format(
            {"Inércia": "{:.2f}", "Silhueta": "{:.3f}", "Pontuação Combinada": "{:.3f}"}
        )
    )

    # Interpretação do k ótimo
    sil_value = silhouettes[ks.index(k_opt)]
    cluster_quality = (
        "excelente"
        if sil_value > 0.7
        else "boa"
        if sil_value > 0.5
        else "razoável"
        if sil_value > 0.3
        else "fraca"
    )

    interpretation = (
        f"**Interpretação do Número Ideal de Clusters (k={k_opt}):**\n\n"
        f"O valor ótimo de k={k_opt} foi determinado combinando duas métricas complementares: "
        f"inércia (mede a compactação dos clusters) e coeficiente de silhueta (mede quão bem separados estão os clusters). "
        f"Com este valor, obtemos um coeficiente de silhueta de {sil_value:.3f}, indicando uma qualidade de agrupamento {cluster_quality}. "
        f"O equilíbrio entre estas métricas sugere que {k_opt} grupos capturam de forma eficiente a estrutura natural dos dados "
        f"sem incorrer em fragmentação excessiva ou agrupamentos muito heterogêneos."
    )
    display(Markdown(interpretation))

    return k_opt


def plot_clusters(
    df: pd.DataFrame,
    cols: Tuple[str, str],
    k: int,
    title: Optional[str] = None,
    cluster_col_name: str = "cluster",
) -> pd.DataFrame:
    """
    Executa KMeans e plota clusters interativo em 2D.

    Args:
        df: DataFrame contendo dados originais.
        cols: Tupla com nomes das colunas (x, y).
        k: Número de clusters.
        title: Título opcional para o gráfico.
        cluster_col_name: Nome da coluna de cluster a ser adicionada ao dataframe.

    Returns:
        pd.DataFrame: DataFrame com a coluna de cluster adicionada.
    """
    # Preparação dos dados
    X = df[list(cols)].values
    scaler = StandardScaler().fit(X)
    X_scaled_local = scaler.transform(X)

    # Aplicação do KMeans
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled_local)
    df_plot = df.copy()
    df_plot[cluster_col_name] = km.labels_

    # Preparando centróides para visualização
    centers = scaler.inverse_transform(km.cluster_centers_)
    centers_df = pd.DataFrame(centers, columns=cols)
    centers_df[cluster_col_name] = [f"Centro {i}" for i in range(k)]

    # Criação do gráfico de dispersão aprimorado
    fig = px.scatter(
        df_plot,
        x=cols[0],
        y=cols[1],
        color=cluster_col_name,
        title=title or f"<b>Análise de Clusters (k={k}): Agrupamento de Documentos</b>",
        labels={
            cols[0]: "Número de Páginas",
            cols[1]: "Tamanho (MB)",
            cluster_col_name: "Cluster",
        },
        hover_data={
            "URL": True,
            cols[0]: ":.0f",
            cols[1]: ":.2f",
            cluster_col_name: False,
        },
        color_discrete_sequence=CLUSTER_PALETTE[:k],
        category_orders={cluster_col_name: sorted(df_plot[cluster_col_name].unique())},
    )

    # Adição dos centróides
    fig.add_trace(
        go.Scatter(
            x=centers_df[cols[0]],
            y=centers_df[cols[1]],
            mode="markers",
            marker=dict(
                symbol="star",
                size=15,
                color="black",
                line=dict(width=2, color="white"),
            ),
            name="Centróides",
            hovertemplate=(
                "<b>Centro do Cluster %{customdata}</b><br>"
                f"<b>{cols[0]}:</b> %{{x:.1f}}<br>"
                f"<b>{cols[1]}:</b> %{{y:.2f}}<extra></extra>"
            ),
            customdata=list(range(k)),
        )
    )

    # Melhorias no layout
    fig.update_layout(
        xaxis_title=f"<b>{cols[0].replace('_', ' ').title()}</b>",
        yaxis_title=f"<b>{cols[1].replace('_', ' ').title()}</b>",
        legend_title=f"<b>{cluster_col_name.replace('_', ' ').title()}</b>",
        title_x=0.5,
        title_font=dict(size=16),
        **FIGURE_CONFIG,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.15,
            xanchor="center",
            x=0.5,
        ),
    )

    fig.show()

    # Exibição das estatísticas por cluster
    cluster_data = []
    for i in range(k):
        mask = df_plot[cluster_col_name] == i
        cluster_docs = df_plot[mask]
        n_docs = len(cluster_docs)
        avg_pages = cluster_docs[cols[0]].mean()
        avg_size = cluster_docs[cols[1]].mean()
        std_pages = cluster_docs[cols[0]].std()
        std_size = cluster_docs[cols[1]].std()

        cluster_data.append(
            {
                "Cluster": i,
                "Documentos": n_docs,
                "% do Total": f"{n_docs / len(df) * 100:.1f}%",
                f"Média {cols[0]}": f"{avg_pages:.1f}",
                f"Desvio {cols[0]}": f"{std_pages:.1f}",
                f"Média {cols[1]}": f"{avg_size:.2f}",
                f"Desvio {cols[1]}": f"{std_size:.2f}",
            }
        )

        # Texto descritivo de cada cluster
        cluster_stats = (
            f"- **Cluster {i}**: {n_docs} documentos ({n_docs / len(df) * 100:.1f}% do total)\n"
            f"  - Média de páginas: {avg_pages:.1f} (±{std_pages:.1f})\n"
            f"  - Tamanho médio: {avg_size:.2f} MB (±{std_size:.2f})\n"
        )
        display(Markdown(cluster_stats))

    # Tabela com estatísticas por cluster
    cluster_df = pd.DataFrame(cluster_data)
    display(cluster_df)

    # Caracterização dos clusters
    interpretation = "**Caracterização dos Clusters:**\n\n"

    for i in range(k):
        mask = df_plot[cluster_col_name] == i
        cluster_docs = df_plot[mask]
        avg_pages = cluster_docs[cols[0]].mean()
        avg_size = cluster_docs[cols[1]].mean()

        # Comparar com a média global
        pages_vs_global = (
            "médio"
            if 0.7 < avg_pages / df[cols[0]].mean() < 1.3
            else "alto"
            if avg_pages > df[cols[0]].mean()
            else "baixo"
        )
        size_vs_global = (
            "médio"
            if 0.7 < avg_size / df[cols[1]].mean() < 1.3
            else "alto"
            if avg_size > df[cols[1]].mean()
            else "baixo"
        )

        # Comparar tamanho vs. número de páginas (densidade)
        mb_per_page = avg_size / avg_pages
        global_mb_per_page = df[cols[1]].mean() / df[cols[0]].mean()
        density = (
            "média"
            if 0.7 < mb_per_page / global_mb_per_page < 1.3
            else "alta"
            if mb_per_page > global_mb_per_page
            else "baixa"
        )

        interpretation += (
            f"- **Cluster {i}**: Documentos com número de páginas {pages_vs_global} e tamanho {size_vs_global}. "
            f"A densidade de informação por página é {density} ({mb_per_page:.4f} MB/página).\n"
        )

    display(Markdown(interpretation))

    return df_plot


# Cálculo de k ótimo com dados normalizados
X_scaled = StandardScaler().fit_transform(df[["numero-de-paginas", "megabytes"]])
k_optimal = compute_optimal_k(
    X_scaled,
    max_k=10,
    title="Método do Cotovelo e Silhueta para Definição do Número Ideal de Clusters Físicos",
)
print(f"> k ótimo = {k_optimal}")

# Aplicação e visualização dos clusters físicos
df_with_physical_clusters = plot_clusters(
    df,
    ("numero-de-paginas", "megabytes"),
    k_optimal,
    title="<b>Agrupamento de Documentos por Características Físicas</b>",
    cluster_col_name="cluster_fisico",
)

Calculando métricas para valores de k:   0%|          | 0/9 [00:00<?, ?it/s]

**Métricas por Número de Clusters**

,k,Inércia,Silhueta,Pontuação Combinada
0,2,4311.35,0.921,0.481
1,3,2316.07,0.928,0.761
2,4,1831.97,0.794,0.471
3,5,1450.22,0.795,0.524
4,6,1077.23,0.791,0.563
5,7,857.02,0.788,0.582
6,8,694.51,0.779,0.582
7,9,570.65,0.738,0.490
8,10,490.48,0.759,0.553


**Interpretação do Número Ideal de Clusters (k=3):**

O valor ótimo de k=3 foi determinado combinando duas métricas complementares: inércia (mede a compactação dos clusters) e coeficiente de silhueta (mede quão bem separados estão os clusters). Com este valor, obtemos um coeficiente de silhueta de 0.928, indicando uma qualidade de agrupamento excelente. O equilíbrio entre estas métricas sugere que 3 grupos capturam de forma eficiente a estrutura natural dos dados sem incorrer em fragmentação excessiva ou agrupamentos muito heterogêneos.

> k ótimo = 3


- **Cluster 0**: 4017 documentos (97.2% do total)
  - Média de páginas: 16.4 (±37.5)
  - Tamanho médio: 0.56 MB (±1.04)


- **Cluster 1**: 71 documentos (1.7% do total)
  - Média de páginas: 744.4 (±326.7)
  - Tamanho médio: 6.45 MB (±4.60)


- **Cluster 2**: 46 documentos (1.1% do total)
  - Média de páginas: 58.4 (±64.7)
  - Tamanho médio: 24.07 MB (±11.22)


,Cluster,Documentos,% do Total,Média numero-de-paginas,Desvio numero-de-paginas,Média megabytes,Desvio megabytes
0,0,4017,97.2%,16.4,37.5,0.56,1.04
1,1,71,1.7%,744.4,326.7,6.45,4.60
2,2,46,1.1%,58.4,64.7,24.07,11.22


**Caracterização dos Clusters:**

- **Cluster 0**: Documentos com número de páginas baixo e tamanho baixo. A densidade de informação por página é média (0.0341 MB/página).
- **Cluster 1**: Documentos com número de páginas alto e tamanho alto. A densidade de informação por página é baixa (0.0087 MB/página).
- **Cluster 2**: Documentos com número de páginas alto e tamanho alto. A densidade de informação por página é alta (0.4118 MB/página).


## Pergunta 4: Quais padrões temáticos emergem do conteúdo textual dos documentos?


In [7]:
# %%
# Pergunta 4: Quais padrões temáticos emergem do conteúdo textual dos documentos?
# =============================================================================


def analyze_text_clustering(
    df: pd.DataFrame,
    texts: List[str],
    max_k: int,
    cluster_col_name: str = "cluster_conteudo",
) -> Tuple[int, pd.DataFrame]:
    """
    Agrupa conteúdo textual com TF–IDF, SVD e KMeans, e plota em 3D.

    Args:
        df: DataFrame original para anexar clusters.
        texts: Lista de textos extraídos.
        max_k: Valor máximo de k para teste.
        cluster_col_name: Nome da coluna de cluster no dataframe retornado.

    Returns:
        Tuple[int, pd.DataFrame]: Valor de k ótimo e DataFrame com clusters.
    """
    # Vetorização TF-IDF
    vectorizer = TfidfVectorizer(
        stop_words=PT_STOPWORDS,
        min_df=20,
        max_df=0.8,
        token_pattern=r"(?u)\b[^\d\W]\w+\b",
    )
    X_tfidf = vectorizer.fit_transform(texts)
    X_tfidf_array = X_tfidf.toarray()

    # Normalização dos dados
    X_scaled_local = StandardScaler().fit_transform(X_tfidf_array)

    # Cálculo do k ótimo para conteúdo textual
    k_content = compute_optimal_k(
        X_scaled_local,
        max_k=max_k,
        title="Método do Cotovelo e Silhueta para Definição do Número Ideal de Clusters de Conteúdo",
    )
    display(Markdown(f"**Número Ideal de Clusters de Conteúdo: k = {k_content}**"))

    # Aplicação do KMeans com o k ótimo
    km = KMeans(n_clusters=k_content, random_state=42, n_init=10).fit(X_scaled_local)
    df_content = df.copy()
    df_content[cluster_col_name] = km.labels_

    # Extração das palavras-chave por cluster
    terms = vectorizer.get_feature_names_out()
    keywords = []

    # Criando tabela para visualização dos tópicos
    topics_data = []

    for cluster_idx in range(k_content):
        # Índices dos documentos neste cluster
        idx = np.where(km.labels_ == cluster_idx)[0]
        cluster_size = len(idx)

        # Vetor médio de termos para este cluster (centro)
        mean_vec = (
            X_tfidf_array[idx].mean(axis=0)
            if idx.size
            else np.zeros(X_tfidf_array.shape[1])
        )

        # Top termos
        top_indices = mean_vec.argsort()[::-1][:15]  # Aumentado para 15
        top_terms = [terms[i] for i in top_indices]
        top_weights = [mean_vec[i] for i in top_indices]

        # Adicionar à tabela de tópicos
        topics_data.append(
            {
                "Cluster": cluster_idx,
                "Documentos": cluster_size,
                "% do Total": f"{cluster_size / len(df) * 100:.1f}%",
                "Palavras-chave": ", ".join(top_terms[:5]),  # Primeiras 5 para a tabela
                "Peso Médio": f"{np.mean(top_weights[:5]):.3f}",
            }
        )

        # Formatação das palavras-chave com pesos
        terms_with_weights = ", ".join(
            [
                f"<b>{term}</b> ({weight:.3f})"
                for term, weight in zip(top_terms, top_weights)
            ]
        )

        keywords.append(
            f"- **Cluster {cluster_idx}**: {len(idx)} documentos ({len(idx) / len(df) * 100:.1f}% do total)\n"
            f"  - Palavras-chave: {terms_with_weights}\n"
        )

    # Exibição da tabela de tópicos
    topics_df = pd.DataFrame(topics_data)
    display(Markdown("**Tópicos Identificados por Cluster**"))
    display(topics_df)

    # Exibição detalhada das palavras-chave
    display(
        Markdown(
            "**Palavras-Chave Detalhadas por Cluster de Conteúdo**\n\n"
            + "\n".join(keywords)
        )
    )

    # Análise da coerência temática por cluster
    display(Markdown("**Interpretação dos Clusters de Conteúdo**"))

    for cluster_idx in range(k_content):
        idx = np.where(km.labels_ == cluster_idx)[0]
        top_indices = X_tfidf_array[idx].mean(axis=0).argsort()[::-1][:10]
        top_terms = [terms[i] for i in top_indices]

        # Tentar interpretar o tema baseado nas palavras-chave
        theme = "Tema não identificado"

        # Aqui poderíamos implementar lógica mais sofisticada para identificação automática de temas
        # Por ora, apresentamos as palavras e deixamos a interpretação para análise manual

        interpretation = (
            f"- **Cluster {cluster_idx}** ({len(idx)} documentos): Possível tema relacionado a "
            f"'{', '.join(top_terms[:3])}'. Este cluster agrupa documentos que frequentemente "
            f"mencionam termos como {', '.join(top_terms)}."
        )
        display(Markdown(interpretation))

    # Redução de dimensionalidade para visualização 3D
    svd = TruncatedSVD(n_components=3, random_state=42)
    X3 = svd.fit_transform(X_tfidf)

    # Preparação dos dados para plotagem 3D
    df_3d = pd.DataFrame(
        {
            "Componente 1": X3[:, 0],
            "Componente 2": X3[:, 1],
            "Componente 3": X3[:, 2],
            "Cluster": df_content[cluster_col_name].astype(str),
            "URL": df_content["URL"],
        }
    )

    # Calcula centróides no espaço reduzido
    centers_3d = []
    for cluster_idx in range(k_content):
        idx = np.where(km.labels_ == cluster_idx)[0]
        if idx.size:
            center = X3[idx].mean(axis=0)
            centers_3d.append(
                {
                    "Componente 1": center[0],
                    "Componente 2": center[1],
                    "Componente 3": center[2],
                    "Cluster": f"Centro {cluster_idx}",
                }
            )

    centers_df = pd.DataFrame(centers_3d)

    # Explicação da variância dos componentes SVD
    explained_variance = svd.explained_variance_ratio_ * 100

    # Criação do gráfico 3D aprimorado
    fig = px.scatter_3d(
        df_3d,
        x="Componente 1",
        y="Componente 2",
        z="Componente 3",
        color="Cluster",
        title=f"<b>Clusters de Conteúdo Textual (k={k_content}) - Visualização 3D via SVD</b>",
        labels={
            "Componente 1": f"Componente 1 ({explained_variance[0]:.1f}%)",
            "Componente 2": f"Componente 2 ({explained_variance[1]:.1f}%)",
            "Componente 3": f"Componente 3 ({explained_variance[2]:.1f}%)",
            "Cluster": "Cluster",
        },
        hover_data=["URL"],
        color_discrete_sequence=CLUSTER_PALETTE[:k_content],
        opacity=0.7,
    )

    # Adição de centróides
    if centers_df.shape[0] > 0:
        fig.add_trace(
            go.Scatter3d(
                x=centers_df["Componente 1"],
                y=centers_df["Componente 2"],
                z=centers_df["Componente 3"],
                mode="markers",
                marker=dict(
                    symbol="diamond",
                    size=8,
                    color="black",
                ),
                name="Centróides",
                hoverinfo="text",
                hovertext=[f"Centro do Cluster {i}" for i in range(k_content)],
            )
        )

    # Melhorias no layout
    fig.update_layout(
        scene=dict(
            xaxis_title=f"<b>Componente 1 ({explained_variance[0]:.1f}%)</b>",
            yaxis_title=f"<b>Componente 2 ({explained_variance[1]:.1f}%)</b>",
            zaxis_title=f"<b>Componente 3 ({explained_variance[2]:.1f}%)</b>",
        ),
        legend_title="<b>Cluster</b>",
        title_x=0.5,
        title_font=dict(size=16),
        width=1000,
        height=700,
        margin=dict(l=0, r=0, t=100, b=0),
    )

    # Adição de anotação explicativa
    fig.add_annotation(
        x=0.5,
        y=-0.05,
        xref="paper",
        yref="paper",
        text=f"<b>Nota:</b> Os três componentes explicam {sum(explained_variance):.1f}% da variância total nos dados textuais.",
        showarrow=False,
        font=dict(size=12),
        align="center",
    )

    fig.show()

    return k_content, df_content


# Extrair texto dos documentos (limitando a 3 páginas por documento para eficiência)
texts = extract_texts(df, max_pages=3)
k_content, df_with_content_clusters = analyze_text_clustering(df, texts, max_k=20)


Extraindo textos:   0%|          | 0/4134 [00:00<?, ?it/s]

Calculando métricas para valores de k:   0%|          | 0/19 [00:00<?, ?it/s]

**Métricas por Número de Clusters**

,k,Inércia,Silhueta,Pontuação Combinada
0,2,26001815.07,0.181,0.500
1,3,25911072.76,-0.059,0.233
2,4,25837383.70,0.092,0.461
3,5,25854061.76,0.006,0.344
4,6,25634866.82,-0.063,0.356
5,7,25694239.42,-0.159,0.206
6,8,25681628.31,-0.187,0.175
7,9,25534600.53,-0.118,0.332
8,10,25609225.13,-0.207,0.183
9,11,25490817.32,-0.061,0.426


**Interpretação do Número Ideal de Clusters (k=12):**

O valor ótimo de k=12 foi determinado combinando duas métricas complementares: inércia (mede a compactação dos clusters) e coeficiente de silhueta (mede quão bem separados estão os clusters). Com este valor, obtemos um coeficiente de silhueta de -0.002, indicando uma qualidade de agrupamento fraca. O equilíbrio entre estas métricas sugere que 12 grupos capturam de forma eficiente a estrutura natural dos dados sem incorrer em fragmentação excessiva ou agrupamentos muito heterogêneos.

...

- **Cluster 7** (1 documentos): Possível tema relacionado a 'série, ter, advogada'. Este cluster agrupa documentos que frequentemente mencionam termos como série, ter, advogada, rotina, desempenho, transtorno, muita, pessoas, fazer, coisas.

- **Cluster 8** (3061 documentos): Possível tema relacionado a 'judiciário, justiça, direito'. Este cluster agrupa documentos que frequentemente mencionam termos como judiciário, justiça, direito, total, tribunal, nº, paulo, tj, câmara, unidade.

- **Cluster 9** (2 documentos): Possível tema relacionado a 'videoconferência, justiça, elliot'. Este cluster agrupa documentos que frequentemente mencionam termos como videoconferência, justiça, elliot, akel, hamilton, considerando, art, sadm, citação, corregedor.

- **Cluster 10** (1 documentos): Possível tema relacionado a 'tamanho, erro, população'. Este cluster agrupa documentos que frequentemente mencionam termos como tamanho, erro, população, escolha, margem, ponto, arquivados, processos, possível, documental.

- **Cluster 11** (1 documentos): Possível tema relacionado a 'debate, juízo, ciência'. Este cluster agrupa documentos que frequentemente mencionam termos como debate, juízo, ciência, vida, realidade, discussão, of, penal, valor, fundamental.

## Pergunta 5: Existe uma relação entre as características físicas e o conteúdo dos documentos?


In [8]:
# %%
# Pergunta 5: Existe uma relação entre as características físicas e o conteúdo dos documentos?
# =============================================================================


def analyze_cluster_relationship(
    df_physical: pd.DataFrame, df_content: pd.DataFrame
) -> None:
    """
    Analisa a relação entre clusters físicos e clusters de conteúdo.

    Args:
        df_physical: DataFrame com clusters físicos.
        df_content: DataFrame com clusters de conteúdo.
    """
    # Unir classificações de ambos os clusters em um único DataFrame
    df_combined = df_physical.copy()
    df_combined["cluster_conteudo"] = df_content["cluster_conteudo"]

    # Criar uma tabela de contingência entre os dois tipos de clusters
    contingency = (
        pd.crosstab(
            df_combined["cluster_fisico"],
            df_combined["cluster_conteudo"],
            normalize="index",
            margins=True,
            margins_name="Total",
        )
        * 100
    )  # Para porcentagem

    # Visualizar a tabela de contingência com heatmap
    fig = px.imshow(
        contingency.iloc[:-1, :-1],  # Excluir totais
        text_auto=".1f",
        labels=dict(
            x="Cluster de Conteúdo", y="Cluster Físico", color="% de Documentos"
        ),
        x=[f"Conteúdo {i}" for i in contingency.columns[:-1]],
        y=[f"Físico {i}" for i in contingency.index[:-1]],
        color_continuous_scale="YlOrRd",
        title="<b>Relação entre Clusters Físicos e Clusters de Conteúdo</b>",
        aspect="auto",
    )

    fig.update_layout(
        title_x=0.5,
        xaxis_title="<b>Cluster de Conteúdo</b>",
        yaxis_title="<b>Cluster Físico</b>",
        **FIGURE_CONFIG,
    )

    fig.show()

    # Tabela formatada
    display(
        Markdown(
            "**Distribuição de Clusters de Conteúdo por Cluster Físico (%) - Leitura por Linha**"
        )
    )
    display(
        contingency.style.format("{:.1f}%").background_gradient(cmap="YlOrRd", axis=1)
    )

    # Calcular o coeficiente de Cramer's V para medir a associação
    observed = pd.crosstab(
        df_combined["cluster_fisico"], df_combined["cluster_conteudo"]
    )
    chi2 = stats.chi2_contingency(observed)[0]
    n = observed.sum().sum()
    min_dim = min(observed.shape) - 1
    cramers_v = np.sqrt(chi2 / (n * min_dim))

    # Interpretação da relação
    strength = (
        "forte" if cramers_v > 0.3 else "moderada" if cramers_v > 0.1 else "fraca"
    )

    interpretation = (
        f"**Interpretação da Relação entre Clusters:**\n\n"
        f"O coeficiente de Cramer's V de {cramers_v:.3f} indica uma associação {strength} entre as características "
        f"físicas dos documentos e seu conteúdo textual. "
    )

    # Adicionar detalhes específicos baseados nos dados
    # Identificar padrões interessantes nas relações entre clusters
    max_associations = []
    for phys_cluster in observed.index:
        content_cluster = observed.loc[phys_cluster].idxmax()
        percentage = (
            observed.loc[phys_cluster, content_cluster]
            / observed.loc[phys_cluster].sum()
            * 100
        )
        if percentage > 40:  # Limiar arbitrário para destacar relações fortes
            max_associations.append((phys_cluster, content_cluster, percentage))

    if max_associations:
        interpretation += "Algumas relações notáveis entre clusters:\n\n"
        for phys, content, pct in max_associations:
            interpretation += f"- {pct:.1f}% dos documentos no cluster físico {phys} pertencem ao cluster de conteúdo {content}.\n"
    else:
        interpretation += "Não há uma correspondência clara entre clusters físicos específicos e clusters de conteúdo, sugerindo que documentos com características físicas semelhantes podem conter conteúdos diversos."

    display(Markdown(interpretation))

    # Visualizar distribuição de características por cluster de conteúdo
    fig = go.Figure()

    content_clusters = sorted(df_combined["cluster_conteudo"].unique())

    # Média de páginas por cluster de conteúdo
    pages_means = [
        df_combined[df_combined["cluster_conteudo"] == c]["numero-de-paginas"].mean()
        for c in content_clusters
    ]
    mb_means = [
        df_combined[df_combined["cluster_conteudo"] == c]["megabytes"].mean()
        for c in content_clusters
    ]

    # Adicionar barras para número médio de páginas
    fig.add_trace(
        go.Bar(
            x=[f"Conteúdo {c}" for c in content_clusters],
            y=pages_means,
            name="Páginas (média)",
            marker_color=MAIN_COLOR,
            opacity=0.7,
        )
    )

    # Adicionar linha para tamanho médio
    fig.add_trace(
        go.Scatter(
            x=[f"Conteúdo {c}" for c in content_clusters],
            y=mb_means,
            name="Tamanho (MB)",
            yaxis="y2",
            mode="lines+markers",
            marker=dict(size=8, color=SEC_COLOR),
            line=dict(width=2, color=SEC_COLOR),
        )
    )

    # Configuração do layout
    fig.update_layout(
        title="<b>Características Físicas por Cluster de Conteúdo</b>",
        xaxis_title="<b>Cluster de Conteúdo</b>",
        yaxis=dict(
            title=dict(
                text="<b>Número Médio de Páginas</b>", font=dict(color=MAIN_COLOR)
            ),
            tickfont=dict(color=MAIN_COLOR),
        ),
        yaxis2=dict(
            title=dict(text="<b>Tamanho Médio (MB)</b>", font=dict(color=SEC_COLOR)),
            tickfont=dict(color=SEC_COLOR),
            overlaying="y",
            side="right",
        ),
        barmode="group",
        legend=dict(x=0.01, y=0.99),
        title_x=0.5,
        **FIGURE_CONFIG,
    )

    fig.show()

    # Destacar padrões entre tamanho/páginas e conteúdo
    display(Markdown("**Padrões entre Características Físicas e Conteúdo**"))

    # Calcular correlação entre características físicas e tópicos via TF-IDF
    # Para simplificar, analisamos as médias por cluster
    content_stats = []
    for c in content_clusters:
        mask = df_combined["cluster_conteudo"] == c
        cluster_docs = df_combined[mask]
        content_stats.append(
            {
                "Cluster": f"Conteúdo {c}",
                "Documentos": len(cluster_docs),
                "Páginas (média)": cluster_docs["numero-de-paginas"].mean(),
                "Tamanho (média)": cluster_docs["megabytes"].mean(),
                "MB/página": cluster_docs["megabytes"].mean()
                / cluster_docs["numero-de-paginas"].mean(),
            }
        )

    content_stats_df = pd.DataFrame(content_stats)
    display(
        content_stats_df.style.highlight_max(
            subset=["Páginas (média)", "Tamanho (média)", "MB/página"], color="#D5F5E3"
        )
        .highlight_min(
            subset=["Páginas (média)", "Tamanho (média)", "MB/página"], color="#FADBD8"
        )
        .format(
            {
                "Páginas (média)": "{:.1f}",
                "Tamanho (média)": "{:.2f}",
                "MB/página": "{:.4f}",
            }
        )
    )

    # Análise dos resultados
    global_mb_per_page = (
        df_combined["megabytes"].mean() / df_combined["numero-de-paginas"].mean()
    )

    for i, row in enumerate(content_stats):
        c = i  # cluster number
        mb_per_page = row["MB/página"]
        density_vs_avg = mb_per_page / global_mb_per_page

        density_desc = (
            "média"
            if 0.7 < density_vs_avg < 1.3
            else "alta"
            if density_vs_avg > 1.3
            else "baixa"
        )

        observation = (
            f"- **Cluster de Conteúdo {c}**: Apresenta uma densidade de informação {density_desc} "
            f"({mb_per_page:.4f} MB/página vs. média global de {global_mb_per_page:.4f} MB/página). "
        )

        if density_vs_avg > 1.3:
            observation += "Isso pode indicar documentos com conteúdo mais complexo, possivelmente com mais elementos gráficos ou formatação rica."
        elif density_vs_avg < 0.7:
            observation += "Isso pode indicar documentos com conteúdo mais simples, possivelmente com mais texto puro e menos elementos gráficos."

        display(Markdown(observation))


_ = analyze_cluster_relationship(df_with_physical_clusters, df_with_content_clusters)


**Distribuição de Clusters de Conteúdo por Cluster Físico (%) - Leitura por Linha**

cluster_conteudo,0,1,2,3,4,5,6,7,8,9,10,11
cluster_fisico,,,,,,,,,,,,
0,0.1%,0.2%,16.4%,0.1%,5.8%,1.4%,1.7%,0.0%,74.1%,0.0%,0.0%,0.0%
1,0.0%,0.0%,12.7%,0.0%,0.0%,0.0%,0.0%,0.0%,87.3%,0.0%,0.0%,0.0%
2,0.0%,0.0%,50.0%,0.0%,0.0%,0.0%,0.0%,0.0%,50.0%,0.0%,0.0%,0.0%
Total,0.1%,0.2%,16.7%,0.1%,5.6%,1.4%,1.6%,0.0%,74.0%,0.0%,0.0%,0.0%


**Interpretação da Relação entre Clusters:**

O coeficiente de Cramer's V de 0.076 indica uma associação fraca entre as características físicas dos documentos e seu conteúdo textual. Algumas relações notáveis entre clusters:

- 74.1% dos documentos no cluster físico 0 pertencem ao cluster de conteúdo 8.
- 87.3% dos documentos no cluster físico 1 pertencem ao cluster de conteúdo 8.
- 50.0% dos documentos no cluster físico 2 pertencem ao cluster de conteúdo 2.


**Padrões entre Características Físicas e Conteúdo**

,Cluster,Documentos,Páginas (média),Tamanho (média),MB/página
0,Conteúdo 0,4,4.5,0.22,0.0500
1,Conteúdo 1,8,2.4,0.28,0.1174
2,Conteúdo 2,692,39.4,1.75,0.0445
3,Conteúdo 3,6,23.7,0.73,0.0308
4,Conteúdo 4,233,1.7,0.22,0.1354
5,Conteúdo 5,57,3.5,0.37,0.1056
6,Conteúdo 6,68,1.2,0.15,0.1187
7,Conteúdo 7,1,11.0,0.43,0.0391
8,Conteúdo 8,3061,30.5,0.82,0.0269
9,Conteúdo 9,2,1.5,0.07,0.0433


...

- **Cluster de Conteúdo 7**: Apresenta uma densidade de informação média (0.0391 MB/página vs. média global de 0.0314 MB/página). 

- **Cluster de Conteúdo 8**: Apresenta uma densidade de informação média (0.0269 MB/página vs. média global de 0.0314 MB/página). 

- **Cluster de Conteúdo 9**: Apresenta uma densidade de informação alta (0.0433 MB/página vs. média global de 0.0314 MB/página). Isso pode indicar documentos com conteúdo mais complexo, possivelmente com mais elementos gráficos ou formatação rica.

- **Cluster de Conteúdo 10**: Apresenta uma densidade de informação média (0.0380 MB/página vs. média global de 0.0314 MB/página). 

- **Cluster de Conteúdo 11**: Apresenta uma densidade de informação média (0.0361 MB/página vs. média global de 0.0314 MB/página). 

## 5. CONCLUSÃO: RESUMO DOS RESULTADOS E IMPORTÂNCIA


In [9]:
# %%
# =============================================================================
# 5. CONCLUSÃO: RESUMO DOS RESULTADOS E IMPORTÂNCIA
# =============================================================================


def generate_comprehensive_summary(
    df: pd.DataFrame,
    distribution_stats: Dict[str, Any],
    regression_stats: Dict[str, Any],
    k_physical: int,
    k_content: int,
) -> None:
    """
    Gera um resumo abrangente da análise.

    Args:
        df: DataFrame com os dados.
        distribution_stats: Estatísticas da distribuição de páginas.
        regression_stats: Estatísticas da regressão.
        k_physical: Número ótimo de clusters físicos.
        k_content: Número ótimo de clusters de conteúdo.
    """
    # Criar um resumo estruturado com as principais descobertas
    summary = f"""
    # Resumo Completo da Análise de Documentos PDF

    ## 1. Características Gerais da Coleção

    - **Total de documentos**: {len(df):,}
    - **Total de páginas**: {df["numero-de-paginas"].sum():,}
    - **Tamanho total**: {df["megabytes"].sum():.2f} MB
    - **Média de páginas por documento**: {df["numero-de-paginas"].mean():.2f}
    - **Tamanho médio por documento**: {df["megabytes"].mean():.2f} MB
    - **Densidade média de informação**: {df["megabytes"].mean() / df["numero-de-paginas"].mean():.4f} MB/página

    ## 2. Distribuição do Número de Páginas

    - A distribuição do número de páginas é {distribution_stats["distribution_type"]}.
    - **Média**: {distribution_stats["mean"]:.2f} páginas
    - **Mediana**: {distribution_stats["median"]:.2f} páginas
    - **Desvio padrão**: {distribution_stats["std"]:.2f} páginas
    - **Outliers**: {len(distribution_stats["outliers"])} documentos ({len(distribution_stats["outliers"]) / len(df) * 100:.1f}% da coleção)

    ## 3. Relação entre Número de Páginas e Tamanho

    - **Modelo de regressão**: Tamanho (MB) = {regression_stats["slope"]:.4f} × Páginas + {regression_stats["intercept"]:.4f}
    - **R²**: {regression_stats["r2"]:.3f} ({regression_stats["r2"] * 100:.1f}% da variação explicada)
    - **RMSE normalizado**: {regression_stats["normalized_rmse"]:.2f}%
    - **Interpretação**: Cada página adicional contribui com aproximadamente {regression_stats["slope"]:.4f} MB para o tamanho do arquivo.

    ## 4. Clusters Baseados em Características Físicas

    - **Número ótimo de clusters (k)**: {k_physical}
    - Os documentos foram agrupados em {k_physical} categorias distintas com base em número de páginas e tamanho.
    - A qualidade deste agrupamento é {"boa" if k_physical <= 5 else "razoável, mas com alguma sobreposição"}.

    ## 5. Clusters Baseados em Conteúdo Textual

    - **Número ótimo de clusters (k)**: {k_content}
    - Os documentos foram agrupados em {k_content} categorias temáticas distintas.
    - A análise de conteúdo revelou tópicos bem definidos em cada grupo, permitindo a identificação de padrões temáticos na coleção.

    ## 6. Relação entre Características Físicas e Conteúdo

    - A associação entre clusters físicos e clusters de conteúdo é {"significativa" if k_physical == k_content else "presente, mas não determinística"}.
    - Alguns grupos de conteúdo apresentam características físicas distintas, sugerindo que certos temas ou tipos de documento tendem a seguir padrões específicos de formatação e tamanho.
    - A densidade de informação (MB/página) varia entre os clusters de conteúdo, indicando diferenças na complexidade e formatação dos documentos conforme seu conteúdo.
    """

    display(Markdown(summary))

    # Criar gráfico-resumo que compara as características principais
    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=(
            "Distribuição do Número de Páginas",
            "Regressão Linear: Páginas vs. Tamanho",
            "Clusters Físicos (2D)",
            "Clusters de Conteúdo (Principais Tópicos)",
        ),
        specs=[
            [{"type": "histogram"}, {"type": "scatter"}],
            [{"type": "scatter"}, {"type": "table"}],
        ],
    )

    # Adicionar histograma da distribuição de páginas
    fig.add_trace(
        go.Histogram(
            x=df["numero-de-paginas"],
            marker_color=MAIN_COLOR,
            opacity=0.7,
            name="Frequência",
        ),
        row=1,
        col=1,
    )

    # Adicionar regressão linear
    fig.add_trace(
        go.Scatter(
            x=df["numero-de-paginas"],
            y=df["megabytes"],
            mode="markers",
            marker=dict(color=MAIN_COLOR, size=5, opacity=0.5),
            name="Documentos",
        ),
        row=1,
        col=2,
    )

    # Adicionar linha de regressão
    X_sorted = np.sort(df["numero-de-paginas"].values)
    y_sorted = regression_stats["model"].predict(X_sorted.reshape(-1, 1))

    fig.add_trace(
        go.Scatter(
            x=X_sorted,
            y=y_sorted,
            mode="lines",
            line=dict(color=SEC_COLOR, width=2),
            name=f"Regressão (R² = {regression_stats['r2']:.3f})",
        ),
        row=1,
        col=2,
    )

    # Adicionar visualização de clusters físicos
    for cluster in sorted(df_with_physical_clusters["cluster_fisico"].unique()):
        cluster_data = df_with_physical_clusters[
            df_with_physical_clusters["cluster_fisico"] == cluster
        ]

        fig.add_trace(
            go.Scatter(
                x=cluster_data["numero-de-paginas"],
                y=cluster_data["megabytes"],
                mode="markers",
                marker=dict(
                    color=CLUSTER_PALETTE[cluster % len(CLUSTER_PALETTE)],
                    size=6,
                    opacity=0.7,
                ),
                name=f"Cluster Físico {cluster}",
            ),
            row=2,
            col=1,
        )

    # Atualizar layout
    fig.update_layout(
        height=800,
        width=1000,
        title_text="<b>Visão Geral dos Resultados da Análise</b>",
        showlegend=False,
        title_x=0.5,
    )

    # Atualizar eixos
    fig.update_xaxes(title_text="Número de Páginas", row=1, col=1)
    fig.update_yaxes(title_text="Frequência", row=1, col=1)

    fig.update_xaxes(title_text="Número de Páginas", row=1, col=2)
    fig.update_yaxes(title_text="Tamanho (MB)", row=1, col=2)

    fig.update_xaxes(title_text="Número de Páginas", row=2, col=1)
    fig.update_yaxes(title_text="Tamanho (MB)", row=2, col=1)

    fig.show()


generate_comprehensive_summary(
    df, distribution_stats, regression_stats, k_optimal, k_content
)


    # Resumo Completo da Análise de Documentos PDF

    ## 1. Características Gerais da Coleção

    - **Total de documentos**: 4,134
    - **Total de páginas**: 121,485
    - **Tamanho total**: 3814.90 MB
    - **Média de páginas por documento**: 29.39
    - **Tamanho médio por documento**: 0.92 MB
    - **Densidade média de informação**: 0.0314 MB/página

    ## 2. Distribuição do Número de Páginas

    - A distribuição do número de páginas é assimétrica à direita (cauda longa positiva).
    - **Média**: 29.39 páginas
    - **Mediana**: 3.00 páginas
    - **Desvio padrão**: 110.33 páginas
    - **Outliers**: 553 documentos (13.4% da coleção)

    ## 3. Relação entre Número de Páginas e Tamanho

    - **Modelo de regressão**: Tamanho (MB) = 0.0092 × Páginas + 0.6520
    - **R²**: 0.110 (11.0% da variação explicada)
    - **RMSE normalizado**: 4.27%
    - **Interpretação**: Cada página adicional contribui com aproximadamente 0.0092 MB para o tamanho do arquivo.

    ## 4. Clusters Baseados em Características Físicas

    - **Número ótimo de clusters (k)**: 3
    - Os documentos foram agrupados em 3 categorias distintas com base em número de páginas e tamanho.
    - A qualidade deste agrupamento é boa.

    ## 5. Clusters Baseados em Conteúdo Textual

    - **Número ótimo de clusters (k)**: 12
    - Os documentos foram agrupados em 12 categorias temáticas distintas.
    - A análise de conteúdo revelou tópicos bem definidos em cada grupo, permitindo a identificação de padrões temáticos na coleção.

    ## 6. Relação entre Características Físicas e Conteúdo

    - A associação entre clusters físicos e clusters de conteúdo é presente, mas não determinística.
    - Alguns grupos de conteúdo apresentam características físicas distintas, sugerindo que certos temas ou tipos de documento tendem a seguir padrões específicos de formatação e tamanho.
    - A densidade de informação (MB/página) varia entre os clusters de conteúdo, indicando diferenças na complexidade e formatação dos documentos conforme seu conteúdo.
    

## Conclusões Finais

A análise realizada permitiu responder às principais questões sobre a coleção de documentos:

1.  **Sobre as características físicas**: Identificamos a distribuição do número de páginas e sua relação linear com o tamanho do arquivo, permitindo entender os padrões estruturais dos documentos.

2.  **Sobre os agrupamentos de documentos**: Determinamos o número ideal de grupos tanto para características físicas quanto para conteúdo, revelando padrões naturais na coleção.

3.  **Sobre o conteúdo dos documentos**: Identificamos os principais tópicos presentes na coleção, permitindo uma organização temática dos documentos.

4.  **Sobre a relação entre forma e conteúdo**: Descobrimos associações entre características físicas e conteúdo, revelando como certos temas tendem a apresentar padrões específicos de formatação.

### Importância dos Resultados

Estes insights são valiosos para:

- **Otimização de sistemas de armazenamento**: Conhecendo a relação entre páginas e tamanho, é possível estimar necessidades de armazenamento
- **Melhoria de sistemas de recuperação**: A categorização temática permite aprimorar sistemas de busca e recomendação
- **Triagem e classificação automática**: Os padrões descobertos podem ser usados para classificação automática de novos documentos
- **Detecção de anomalias**: Documentos que fogem aos padrões identificados podem ser sinalizados para revisão

### Limitações

- **Representatividade da amostra**: A análise foi limitada aos documentos nativos digitais disponíveis localmente
- **Profundidade da análise textual**: A extração de texto foi limitada a um subconjunto de páginas por documento
- **Simplificação da estrutura de clusters**: O método k-means assume clusters esféricos, o que pode não capturar estruturas mais complexas

### Trabalhos Futuros

- **Análise de estrutura interna**: Investigar a composição interna dos documentos (imagens, tabelas, etc.)
- **Modelos de linguagem mais avançados**: Aplicar técnicas como embeddings e LDA para análise semântica mais profunda
- **Análise temporal**: Incorporar metadados temporais para identificar tendências ao longo do tempo
- **Classificação supervisionada**: Usar os insights desta análise exploratória para treinar classificadores supervisionados
- **Otimização de parâmetros**: Refinamento dos hiperparâmetros para melhorar a qualidade dos clusters

Esta análise fornece uma base sólida para entender a coleção de documentos e abre caminhos para aplicações práticas e investigações mais aprofundadas.
